# 🛒 Market Basket Analysis — Analyse du Panier

Ce notebook reproduit en Python les étapes DAX décrites dans `Basket_analysis_mining.md`.

**Objectif :** Calculer les métriques **Support**, **Confidence** et **Lift** pour toutes les paires de produits.

**Source de données :** Le fichier `groceries_long.csv` est généré à partir de la base `grocery_db` (dataset Kaggle).  
Chaque `transaction_id` correspond au regroupement `(customerid, date)` — chaque client par jour constitue un panier.  
Cette approche est identique à celle utilisée dans le backend SQL (matérialisation `mv_daily_baskets`).

**Correction :** Les noms de produits contenant des caractères spéciaux (comme `#`) sont nettoyés automatiquement.

## Étape 1 — Import des bibliothèques

Comme en DAX on utilise `DISTINCTCOUNT`, `VALUES`, `INTERSECT`, etc. — en Python on utilise **pandas**, **numpy** et **itertools**.

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import defaultdict
import re

print("Bibliothèques importées avec succès ✓")

Bibliothèques importées avec succès ✓


## Étape 2 — Vérifications de base (équivalent DAX)

**DAX d'origine :**
```dax
Total Transactions = DISTINCTCOUNT(DF_Basket[id_transaction])
NbProduits = DISTINCTCOUNT(DF_Basket[productname])
```

**Données :** ~6,69M lignes, 6,55M paniers uniques (customerid|date), 452 produits.  
~132K paniers multi-produits (2%) — le groupement par `(customerid, date)` permet de capturer les associations d'achat.

On reproduit ces mesures avec pandas.

In [2]:
# Chargement des données
df = pd.read_csv('groceries_long.csv')

# Nettoyage : suppression uniquement des caractères vraiment problématiques
# On garde /, -, etc. qui sont des noms de produits valides
def clean_product_name(name):
    name = str(name).strip()
    # On ne supprime QUE # et ` (backtick) qui posent problème dans les f-strings Python
    name = re.sub(r'[#`]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

df['product_name'] = df['product_name'].apply(clean_product_name)

# Supprimer les lignes avec des noms de produits vides après nettoyage
df = df[df['product_name'] != ''].copy()

# Équivalent DAX : DISTINCTCOUNT(id_transaction)
total_transactions = df['transaction_id'].nunique()
print(f"Total Transactions (DISTINCTCOUNT) = {total_transactions}")

# Équivalent DAX : DISTINCTCOUNT(productname)
nb_products = df['product_name'].nunique()
print(f"Nb Produits (DISTINCTCOUNT) = {nb_products}")

print(f"\nLignes totales = {len(df)}")
print(f"Produits moy. par panier = {len(df) / total_transactions:.2f}")

# Aperçu des données
df.head(10)

Total Transactions (DISTINCTCOUNT) = 6556433
Nb Produits (DISTINCTCOUNT) = 452

Lignes totales = 6690599
Produits moy. par panier = 1.02


,transaction_id,product_name
0,31108|2022-09-18,French Pastry - Mini Chocolate
1,19124|2021-06-28,Olives - Kalamata
2,20421|2018-03-17,Wine - Sogrape Mateus Rose
3,14125|2019-01-26,Flour - Whole Wheat
4,86300|2018-07-03,"Lamb - Whole, Fresh"
5,34530|2018-01-06,Nantuket Peach Orange
6,90109|2022-03-08,Blackberries
7,94315|2018-12-12,Smirnoff Green Apple Twist
8,77947|2019-10-21,Mangoes
9,58990|2022-11-23,Beets - Mini Golden


## Étape 3 — Créer la table des couples produits (équivalent DAX)

**DAX d'origine :**
```dax
Basket Analysis =
FILTER(
    CROSSJOIN(
        SELECTCOLUMNS(VALUES(DF_Basket[productname]), "Product1", DF_Basket[productname]),
        SELECTCOLUMNS(VALUES(DF_Basket[productname]), "Product2", DF_Basket[productname])
    ),
    [Product1] > [Product2]
)
```

En Python, on génère toutes les combinaisons uniques de 2 produits par transaction avec `itertools.combinations`.

In [3]:
# Grouper les produits par transaction (équivalent VALUES + CROSSJOIN)
transactions = df.groupby('transaction_id')['product_name'].apply(list).to_dict()

# Générer toutes les paires uniques
# Équivalent DAX : CROSSJOIN + FILTER([Product1] > [Product2])
# utilisation de combinations() garantit :
#   - Chaque paire n'apparaît qu'une seule fois (pas de A→B ET B→A)
#   - Order alphabétique : Product1 < Product2
pair_counts = defaultdict(int)

for trans_id, products in transactions.items():
    unique_products = sorted(set(products))
    for a, b in combinations(unique_products, 2):
        pair_counts[(a, b)] += 1

print(f"Nombre total de paires uniques : {len(pair_counts)}")
print(f"\nExemple de paires trouvées :")
for (a, b), cnt in list(pair_counts.items())[:5]:
    print(f"  {a:25s} → {b:25s} -> {cnt} transactions")

# Vérification anti-doublons : s'assurer qu'aucune paire inverse (B,A) n'existe
paires_inverses = 0
for (a, b) in pair_counts:
    if (b, a) in pair_counts:
        paires_inverses += 1
        if paires_inverses <= 3:
            print(f"\n  ⚠ Paire inverse trouvée : ({a},{b}) et ({b},{a})")
            
print(f"\nPaires inverses (B,A) détectées : {paires_inverses}")
print(f"Résultat : {'✓ AUCUNE - combinations() garantit l\'unicité' if paires_inverses == 0 else '✗ ERREUR'}")

Nombre total de paires uniques : 75020

Exemple de paires trouvées :
  Olives - Kalamata         → Pail For Lid 1537         -> 2 transactions
  Chicken - Soup Base       → Lettuce - Frisee          -> 2 transactions
  Mangoes                   → Mushroom - Trumpet, Dry   -> 3 transactions
  Ecolab - Mikroklene 4/4 L → Wasabi Powder             -> 2 transactions
  Baking Powder             → Dried Figs                -> 4 transactions

Paires inverses (B,A) détectées : 0
Résultat : ✓ AUCUNE - combinations() garantit l'unicité


## Étape 4 — Calcul du Support (équivalent DAX)

**DAX d'origine :**
```dax
Support =
VAR TransP1 = CALCULATETABLE(VALUES(DF_Basket[id_transaction]), DF_Basket[productname] = Prod1)
VAR TransP2 = CALCULATETABLE(VALUES(DF_Basket[id_transaction]), DF_Basket[productname] = Prod2)
VAR BothTrans = INTERSECT(TransP1, TransP2)
RETURN DIVIDE(COUNTROWS(BothTrans), DISTINCTCOUNT(DF_Basket[id_transaction]), 0)
```

**Formule :** Support(X,Y) = Nb transactions contenant X ET Y / Nb total de transactions

In [4]:
# Compter les occurrences de chaque produit (équivalent VALUES + COUNTROWS)
product_counts = df['product_name'].value_counts()
n_transactions = total_transactions

# Calcul du Support pour chaque paire
# Support = COUNTROWS(BothTrans) / TotalTrans
# IMPORTANT : on garde la valeur EXACTE (pas d'arrondi) pour ne pas fausser le Lift
basket_data = []
for (a, b), count in pair_counts.items():
    support = count / n_transactions
    basket_data.append({
        'Product1': a,
        'Product2': b,
        'Basket': f"{a} → {b}",  # flèche pour montrer le sens de l'association
        'Support': support,  # valeur exacte en décimal (0-1) pour les calculs
        'Support %': round(support * 100, 2),  # valeur en pourcentage (0-100) pour l'affichage
        'NbTransactions': count
    })

# Création du DataFrame (équivalent de la table "Basket Analysis")
basket_df = pd.DataFrame(basket_data)
print(f"Table Basket Analysis créée : {len(basket_df)} lignes")
print(f"\nTop 10 des paires par Support :")
basket_df.sort_values('Support', ascending=False).head(10)

Table Basket Analysis créée : 75020 lignes

Top 10 des paires par Support :


,Product1,Product2,Basket,Support,Support %,NbTransactions
6515,Beef Ground Medium,Broom - Corn,Beef Ground Medium → Broom - Corn,0.000002,0.0,12
499,Beer - Blue,Lettuce - Spring Mix,Beer - Blue → Lettuce - Spring Mix,0.000001,0.0,9
10343,Ice Cream Bar - Oreo Cone,Veal - Slab Bacon,Ice Cream Bar - Oreo Cone → Veal - Slab Bacon,0.000001,0.0,8
20749,"Beans - Kidney, Canned",Phyllo Dough,"Beans - Kidney, Canned → Phyllo Dough",0.000001,0.0,8
20726,Kiwi,"Mushroom - Trumpet, Dry","Kiwi → Mushroom - Trumpet, Dry",0.000001,0.0,8
16020,Dc Hikiage Hira Huba,Milk Powder,Dc Hikiage Hira Huba → Milk Powder,0.000001,0.0,8
17857,Pastry - Raisin Muffin - Mini,Tea - Herbal Sweet Dreams,Pastry - Raisin Muffin - Mini → Tea - Herbal S...,0.000001,0.0,8
9134,Artichokes - Jerusalem,Cheese - Cambozola,Artichokes - Jerusalem → Cheese - Cambozola,0.000001,0.0,7
10685,Pears - Bosc,"Soup - Campbells, Lentil","Pears - Bosc → Soup - Campbells, Lentil",0.000001,0.0,7
10885,Cookie - Dough Variety,"Potatoes - Instant, Mashed","Cookie - Dough Variety → Potatoes - Instant, M...",0.000001,0.0,7


## Étape 5 — Calcul de la Confidence (P1 → P2)

**DAX d'origine :**
```dax
Confidence P1 =
VAR TransP1 = CALCULATETABLE(VALUES(DF_Basket[id_transaction]), DF_Basket[productname] = Prod1)
VAR BothTrans = INTERSECT(TransP1, CALCULATETABLE(...))
RETURN DIVIDE(NbBoth, NbP1, 0)
```

**Formule :** Confidence(P1→P2) = Support(P1,P2) / Support(P1) = Nb(P1∩P2) / Nb(P1)

In [5]:
# Calcul de Confidence P1->P2 et P2->P1
# Confidence P1->P2 = Nb(P1 et P2) / Nb(P1)
# Confidence P2->P1 = Nb(P1 et P2) / Nb(P2)

confidence_p1 = []
confidence_p2 = []
confidence_p1_pct = []
confidence_p2_pct = []

for _, row in basket_df.iterrows():
    prod1 = row['Product1']
    prod2 = row['Product2']
    nb_both = row['NbTransactions']
    
    count_p1 = product_counts.get(prod1, 0)
    count_p2 = product_counts.get(prod2, 0)
    
    # Équivalent DAX : DIVIDE(NbBoth, NbP1, 0)
    # On garde la valeur exacte en décimal (0-1) pour les calculs (Lift)
    conf1 = nb_both / count_p1 if count_p1 > 0 else 0
    conf2 = nb_both / count_p2 if count_p2 > 0 else 0
    
    confidence_p1.append(conf1)       # valeur exacte pour les calculs
    confidence_p2.append(conf2)       # valeur exacte pour les calculs
    confidence_p1_pct.append(round(conf1 * 100, 2))  # pourcentage pour l'affichage
    confidence_p2_pct.append(round(conf2 * 100, 2))  # pourcentage pour l'affichage

basket_df['Confidence P1'] = confidence_p1
basket_df['Confidence P2'] = confidence_p2
basket_df['Confidence P1 %'] = confidence_p1_pct
basket_df['Confidence P2 %'] = confidence_p2_pct

print("Confidence calculée pour toutes les paires ✓")
print(f"\nTop 10 par Confidence P1->P2 :")
basket_df.sort_values('Confidence P1', ascending=False).head(10)

Confidence calculée pour toutes les paires ✓

Top 10 par Confidence P1->P2 :


,Product1,Product2,Basket,Support,Support %,NbTransactions,Confidence P1,Confidence P2,Confidence P1 %,Confidence P2 %
6515,Beef Ground Medium,Broom - Corn,Beef Ground Medium → Broom - Corn,0.000002,0.0,12,0.000812,0.000812,0.08,0.08
499,Beer - Blue,Lettuce - Spring Mix,Beer - Blue → Lettuce - Spring Mix,0.000001,0.0,9,0.000612,0.000607,0.06,0.06
16020,Dc Hikiage Hira Huba,Milk Powder,Dc Hikiage Hira Huba → Milk Powder,0.000001,0.0,8,0.000546,0.000543,0.05,0.05
20749,"Beans - Kidney, Canned",Phyllo Dough,"Beans - Kidney, Canned → Phyllo Dough",0.000001,0.0,8,0.000543,0.000540,0.05,0.05
20726,Kiwi,"Mushroom - Trumpet, Dry","Kiwi → Mushroom - Trumpet, Dry",0.000001,0.0,8,0.000542,0.000542,0.05,0.05
17857,Pastry - Raisin Muffin - Mini,Tea - Herbal Sweet Dreams,Pastry - Raisin Muffin - Mini → Tea - Herbal S...,0.000001,0.0,8,0.000540,0.000539,0.05,0.05
10343,Ice Cream Bar - Oreo Cone,Veal - Slab Bacon,Ice Cream Bar - Oreo Cone → Veal - Slab Bacon,0.000001,0.0,8,0.000535,0.000543,0.05,0.05
2179,Garbage Bags - Clear,Squid - Tubes / Tenticles 10/20,Garbage Bags - Clear → Squid - Tubes / Tenticl...,0.000001,0.0,7,0.000481,0.000480,0.05,0.05
10885,Cookie - Dough Variety,"Potatoes - Instant, Mashed","Cookie - Dough Variety → Potatoes - Instant, M...",0.000001,0.0,7,0.000477,0.000471,0.05,0.05
53413,Curry Paste - Madras,Wine - Pinot Noir Latour,Curry Paste - Madras → Wine - Pinot Noir Latour,0.000001,0.0,7,0.000476,0.000476,0.05,0.05


## Étape 6 — Calcul du Lift (équivalent DAX)

**DAX d'origine :**
```dax
Lift =
VAR SupportP1 = DIVIDE(COUNTROWS(TransP1), TotalTrans, 0)
VAR SupportP2 = DIVIDE(COUNTROWS(TransP2), TotalTrans, 0)
VAR SupportBoth = DIVIDE(COUNTROWS(BothTrans), TotalTrans, 0)
RETURN DIVIDE(SupportBoth, SupportP1 * SupportP2, 0)
```

**Formule :** Lift = Support(X,Y) / (Support(X) × Support(Y))

| Lift | Interprétation |
|------|---------------|
| < 1 | Corrélation négative |
| = 1 | Indépendant |
| > 1 | Association positive |

In [6]:
# Calcul du Lift
# Lift = Support(X,Y) / (Support(X) * Support(Y))
# Support(X) = count(X) / TotalTrans
# IMPORTANT : on garde les valeurs EXACTES sans arrondir dans la boucle
# pour ne pas fausser le classement et les filtres

lifts = []
for _, row in basket_df.iterrows():
    prod1 = row['Product1']
    prod2 = row['Product2']
    support_ab = row['Support']
    
    count_p1 = product_counts.get(prod1, 0)
    count_p2 = product_counts.get(prod2, 0)
    
    support_a = count_p1 / n_transactions
    support_b = count_p2 / n_transactions
    
    # Équivalent DAX : DIVIDE(SupportBoth, SupportP1 * SupportP2, 0)
    if support_a * support_b > 0:
        lift = support_ab / (support_a * support_b)
    else:
        lift = 0
    
    lifts.append(lift)  # valeur exacte, pas d'arrondi !

basket_df['Lift'] = lifts

print("Lift calculé pour toutes les paires ✓")
print(f"\nTop 20 des associations (Lift >= 1.5, trié par Lift) :")

# Filtrer Lift >= 1.5 comme recommandé dans le document
top_lift = basket_df[basket_df['Lift'] >= 1.5].sort_values('Lift', ascending=False)
display(top_lift.head(20))

Lift calculé pour toutes les paires ✓

Top 20 des associations (Lift >= 1.5, trié par Lift) :


,Product1,Product2,Basket,Support,Support %,NbTransactions,Confidence P1,Confidence P2,Confidence P1 %,Confidence P2 %,Lift


## Étape 7 — Analyse et interprétation des résultats

On applique les seuils recommandés par le document :
- **Lift ≥ 1,5** → association intéressante
- **Support ≥ 0,3%** → fréquence minimale pour être pertinent

In [7]:
# Analyse des résultats avec les seuils du document

print("=== RÉPARTITION DES LIFT ===")
print(f"  Négatif (< 1)          : {len(basket_df[basket_df['Lift'] < 1]):5d} paires")
print(f"  Faible (1 - 1.5)       : {len(basket_df[(basket_df['Lift'] >= 1) & (basket_df['Lift'] < 1.5)]):5d} paires")
print(f"  Bonne (1.5 - 2)        : {len(basket_df[(basket_df['Lift'] >= 1.5) & (basket_df['Lift'] < 2)]):5d} paires")
print(f"  Forte (2 - 5)          : {len(basket_df[(basket_df['Lift'] >= 2) & (basket_df['Lift'] < 5)]):5d} paires")
print(f"  Très forte (5 - 10)    : {len(basket_df[(basket_df['Lift'] >= 5) & (basket_df['Lift'] < 10)]):5d} paires")
print(f"  Exceptionnelle (> 10)  : {len(basket_df[basket_df['Lift'] >= 10]):5d} paires")

print("\n=== TOP 10 ASSOCIATIONS PERTINENTES (Support >= 0.3%, Lift >= 1.5) ===")
pertinent = basket_df[(basket_df['Support'] >= 0.003) & (basket_df['Lift'] >= 1.5)]
pertinent = pertinent.sort_values('Lift', ascending=False)
# Afficher Support % et Confidence % avec le symbole
affichage = pertinent[['Basket', 'Support %', 'Confidence P1 %', 'Confidence P2 %', 'Lift']].head(10).copy()
affichage.columns = ['Basket', 'Support %', 'Confidence P1 %', 'Confidence P2 %', 'Lift']
display(affichage)

print("\n=== TOP 10 PAR CONFIDENCE (Support >= 0.5%) ===")
top_conf = basket_df[basket_df['Support'] >= 0.005].sort_values('Confidence P1', ascending=False)
affichage2 = top_conf[['Basket', 'Support %', 'Confidence P1 %', 'Lift']].head(10).copy()
affichage2.columns = ['Basket', 'Support %', 'Confidence P1 %', 'Lift']
display(affichage2)

=== RÉPARTITION DES LIFT ===
  Négatif (< 1)          : 75020 paires
  Faible (1 - 1.5)       :     0 paires
  Bonne (1.5 - 2)        :     0 paires
  Forte (2 - 5)          :     0 paires
  Très forte (5 - 10)    :     0 paires
  Exceptionnelle (> 10)  :     0 paires

=== TOP 10 ASSOCIATIONS PERTINENTES (Support >= 0.3%, Lift >= 1.5) ===


,Basket,Support %,Confidence P1 %,Confidence P2 %,Lift



=== TOP 10 PAR CONFIDENCE (Support >= 0.5%) ===


,Basket,Support %,Confidence P1 %,Lift


## Étape 8 — Export du fichier CSV (table Basket Analysis)

On exporte la table complète `Basket Analysis` au format CSV, comme demandé.

In [8]:
# Export de la table Basket Analysis complète en CSV
# Format compatible avec la table PostgreSQL basket_analysis_results
# Colonnes en minuscules avec valeurs brutes (décimales) + pourcentages

export_df = basket_df[[
    'Product1', 'Product2', 'Basket', 'Support', 'Confidence P1',
    'Confidence P2', 'Lift'
]].copy()

# Ajouter les colonnes supplémentaires pour la base de données
export_df['Support_Pct'] = (basket_df['Support'] * 100).round(4)
export_df['Confidence_P1_Pct'] = (basket_df['Confidence P1'] * 100).round(4)
export_df['Confidence_P2_Pct'] = (basket_df['Confidence P2'] * 100).round(4)
export_df['NbTransactions'] = basket_df['NbTransactions']

# Renommer pour correspondre au schéma PostgreSQL
export_df.columns = ['product1', 'product2', 'basket_label', 'support',
                     'confidence_p1', 'confidence_p2', 'lift',
                     'support_pct', 'confidence_p1_pct', 'confidence_p2_pct',
                     'nb_transactions']

# Arrondir lift à 4 décimales
export_df['lift'] = export_df['lift'].round(4)

# Sauvegarde en CSV (racine + data/csv_exports/)
import os
output_path = 'basket_analysis_results.csv'
export_df.to_csv(output_path, index=False, encoding='utf-8-sig')

# Copie dans data/csv_exports/ si le dossier existe
csv_exports_dir = 'data/csv_exports'
if os.path.isdir(csv_exports_dir):
    export_df.to_csv(os.path.join(csv_exports_dir, 'basket_analysis_results.csv'), index=False, encoding='utf-8-sig')
    print(f"✓ Copié vers {csv_exports_dir}/")
else:
    print(f"⚠ Dossier {csv_exports_dir} non trouvé")

print(f"Fichier exporté : {output_path}")
print(f"Lignes exportées : {len(export_df)}")
print(f"Colonnes : {list(export_df.columns)}")

# Aperçu du fichier exporté
print("\nAperçu des 5 premières lignes du CSV :")
export_df.head()

Fichier exporté : basket_analysis_results.csv
Lignes exportées : 75020
Colonnes : ['Product1', 'Product2', 'Basket', 'Support %', 'Confidence P1 %', 'Confidence P2 %', 'Lift']

Aperçu des 5 premières lignes du CSV :


,Product1,Product2,Basket,Support %,Confidence P1 %,Confidence P2 %,Lift
0,Olives - Kalamata,Pail For Lid 1537,Olives - Kalamata → Pail For Lid 1537,0.0,0.01,0.01,0.0606
1,Chicken - Soup Base,Lettuce - Frisee,Chicken - Soup Base → Lettuce - Frisee,0.0,0.01,0.01,0.0603
2,Mangoes,"Mushroom - Trumpet, Dry","Mangoes → Mushroom - Trumpet, Dry",0.0,0.02,0.02,0.0902
3,Ecolab - Mikroklene 4/4 L,Wasabi Powder,Ecolab - Mikroklene 4/4 L → Wasabi Powder,0.0,0.01,0.01,0.0596
4,Baking Powder,Dried Figs,Baking Powder → Dried Figs,0.0,0.03,0.03,0.1168


## ✅ Résumé

Le notebook a reproduit fidèlement le code DAX en Python :

| Métrique | DAX | Python |
|----------|-----|--------|
| Total Transactions | `DISTINCTCOUNT(id_transaction)` | `df['transaction_id'].nunique()` |
| Table des couples | `CROSSJOIN + FILTER` | `itertools.combinations()` |
| Support | `COUNTROWS(INTERSECT(...)) / TotalTrans` | `count_pair / n_transactions` |
| Confidence | `DIVIDE(NbBoth, NbP1, 0)` | `nb_both / count_p1` |
| Lift | `DIVIDE(SupportBoth, SupportP1 * SupportP2, 0)` | `support_ab / (support_a * support_b)` |

**Correction appliquée :** Les caractères problématiques (`#`, `` ` ``, etc.) dans les noms de produits sont nettoyés automatiquement.

**Fichier généré :** `basket_analysis_results.csv`

## 🔬 Validation croisée — Vérification de l'intégrité des calculs

In [10]:
# Validation croisée des calculs
# Utilisation de produits réels du dataset (≠ 'whole milk' du dataset Groceries classique)
print("=" * 60)
print("🔬 VALIDATION CROISÉE")
print("=" * 60)

# Trouver le produit le plus fréquent comme référence pour les tests
produit_test = product_counts.idxmax()
count_p = product_counts[produit_test]
support_calcule = count_p / n_transactions
print(f"\n1. Support individuel de '{produit_test}' (produit le + fréquent):")
print(f"   count = {count_p}, TotalTrans = {n_transactions}")
print(f"   Support = {count_p}/{n_transactions} = {support_calcule:.6f} ({support_calcule*100:.2f}%)")

# Test 2 : Vérifier la relation Support * TotalTrans = NbTransactions pour une paire
# Prendre la paire avec le plus de co-occurrences
top_pair_idx = basket_df['NbTransactions'].idxmax()
paire_test = basket_df.loc[top_pair_idx]
print(f"\n2. Vérification paire '{paire_test['Basket']}':")
print(f"   Support = {paire_test['Support']:.8f}")
print(f"   NbTransactions = {paire_test['NbTransactions']}")
print(f"   Support * TotalTrans = {paire_test['Support'] * n_transactions:.2f}")
print(f"   Correspond à NbTransactions ? {'✓ OUI' if abs(paire_test['Support'] * n_transactions - paire_test['NbTransactions']) < 0.01 else '✗ NON'}")

# Test 3 : Vérifier que Confidence P1 = NbBoth / Count(P1)
prod1 = paire_test['Product1']
prod2 = paire_test['Product2']
count_p1 = product_counts[prod1]
conf_attendue = paire_test['NbTransactions'] / count_p1
print(f"\n3. Vérification Confidence P1 ('{prod1}' -> '{prod2}'):")
print(f"   NbBoth = {paire_test['NbTransactions']}, Count({prod1}) = {count_p1}")
print(f"   Confidence = {paire_test['NbTransactions']}/{count_p1} = {conf_attendue:.6f}")
print(f"   Valeur dans la table = {paire_test['Confidence P1']:.6f}")
print(f"   Correspond ? {'✓ OUI' if abs(conf_attendue - paire_test['Confidence P1']) < 0.0001 else '✗ NON'}")

# Test 4 : Vérifier que Lift = Support_AB / (Support_A * Support_B)
support_a = count_p1 / n_transactions
support_b = product_counts[prod2] / n_transactions
support_ab = paire_test['Support']
lift_attendu = support_ab / (support_a * support_b) if support_a * support_b > 0 else 0
print(f"\n4. Vérification Lift ('{prod1}' & '{prod2}'):")
print(f"   Support(A) = {support_a:.6f}, Support(B) = {support_b:.6f}")
print(f"   Support(AB) = {support_ab:.8f}")
print(f"   Lift = {support_ab:.8f} / ({support_a:.6f} * {support_b:.6f}) = {lift_attendu:.6f}")
print(f"   Valeur dans la table = {paire_test['Lift']:.6f}")
print(f"   Correspond ? {'✓ OUI' if abs(lift_attendu - paire_test['Lift']) < 0.001 else '✗ NON'}")

# Test 5 : Vérifier l'ordre alphabétique Product1 < Product2
mauvais_ordre = basket_df[basket_df['Product1'] >= basket_df['Product2']]
print(f"\n5. Vérification ordre alphabétique (Product1 < Product2) :")
print(f"   Paires mal ordonnées : {len(mauvais_ordre)}")
print(f"   Résultat : {'✓ PARFAIT' if len(mauvais_ordre) == 0 else '✗ ERREUR'}")

# Test 6 : Vérifier qu'il n'y a pas de doublons de paires
doublons = basket_df.duplicated(subset=['Product1', 'Product2']).sum()
print(f"\n6. Vérification des doublons (même paire en double) :")
print(f"   Paires dupliquées : {doublons}")
print(f"   Résultat : {'✓ PARFAIT' if doublons == 0 else '✗ ERREUR'}")

# Test 7 : Vérification spécifique des paires inverses (A→B et B→A)
print(f"\n7. Vérification des paires inverses (A→B et B→A) :")
pairs_set = set(zip(basket_df['Product1'], basket_df['Product2']))
paires_inverses = 0
exemples_inverses = []
for p1, p2 in pairs_set:
    if (p2, p1) in pairs_set:
        paires_inverses += 1
        if len(exemples_inverses) < 3:
            exemples_inverses.append(f"({p1}→{p2}) et ({p2}→{p1})")
print(f"   Paires inverses trouvées : {paires_inverses}")
if exemples_inverses:
    for ex in exemples_inverses:
        print(f"   ⚠ {ex}")
print(f"   Résultat : {'✓ PARFAIT - aucune paire inverse' if paires_inverses == 0 else f'✗ {paires_inverses} paires inverses détectées'}")

# Test 8 : Vérifier la cohérence des totaux
total_paires_calcule = len(pair_counts)
print(f"\n8. Vérification cohérence totale :")
print(f"   Paires dans pair_counts : {total_paires_calcule}")
print(f"   Lignes dans basket_df   : {len(basket_df)}")
print(f"   Résultat : {'✓ PARFAIT' if total_paires_calcule == len(basket_df) else '✗ ERREUR'}")

# Test 9 : Vérifier que Support * TotalTrans = un nombre entier de transactions
support_check = basket_df['Support'].head(100) * n_transactions
support_entier = all(abs(support_check.round(0) - support_check) < 0.01)
print(f"\n9. Vérification Support * TotalTrans = entier (100 premières paires) :")
print(f"   Résultat : {'✓ PARFAIT' if support_entier else '⚠ Légères imprécisions flottantes (normales)'}")

print(f"\n{'='*60}")
print("✅ VALIDATION TERMINÉE")
print(f"{'='*60}")

🔬 VALIDATION CROISÉE

1. Support individuel de 'Longos - Chicken Wings' (produit le + fréquent):
   count = 15215, TotalTrans = 6556433
   Support = 15215/6556433 = 0.002321 (0.23%)

2. Vérification paire 'Beef Ground Medium → Broom - Corn':
   Support = 0.00000183
   NbTransactions = 12
   Support * TotalTrans = 12.00
   Correspond à NbTransactions ? ✓ OUI

3. Vérification Confidence P1 ('Beef Ground Medium' -> 'Broom - Corn'):
   NbBoth = 12, Count(Beef Ground Medium) = 14779
   Confidence = 12/14779 = 0.000812
   Valeur dans la table = 0.000812
   Correspond ? ✓ OUI

4. Vérification Lift ('Beef Ground Medium' & 'Broom - Corn'):
   Support(A) = 0.002254, Support(B) = 0.002255
   Support(AB) = 0.00000183
   Lift = 0.00000183 / (0.002254 * 0.002255) = 0.360139
   Valeur dans la table = 0.360139
   Correspond ? ✓ OUI

5. Vérification ordre alphabétique (Product1 < Product2) :
   Paires mal ordonnées : 0
   Résultat : ✓ PARFAIT

6. Vérification des doublons (même paire en double) :
   Pa